# Ejercicio KNN + Pipeline — Regresión de precios de vivienda

Guía paso a paso para trabajar **en clase**. Cada paso tiene:

1. Una celda de **markdown** con la explicación de qué hacer.
2. Una celda de **código** con un esqueleto (`# TODO`) que ustedes deben completar.

En 3 puntos del notebook (marcados como **Checkpoint**) hay una celda con `assert` que deben pasar antes de seguir. Si la celda corre sin error e imprime `✅`, van por buen camino. Si falla, lean el mensaje del `assert`: les dice qué revisar.

No modifiquen las celdas de checkpoint.

## Paso 0 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42


## Paso 1 — Carga y exploración de datos

- Carga `data/house_price_regression_dataset.csv` en un DataFrame llamado `df`.
- Revisa `shape`, `dtypes`, nulos (`isnull().sum()`) y `describe()`.
- Confirma que no haya valores nulos.

In [ ]:
# TODO: carga el csv en df

# TODO: explora el dataframe (shape, dtypes, nulos, describe)


## Paso 2 — ¿Por qué es un problema de regresión?

Mira la columna `House_Price`: ¿cuántos valores distintos tiene?, ¿qué tipo de dato es?

**Responde en esta celda (mínimo 3 líneas):** ¿por qué este problema es de regresión y no de clasificación? ¿Qué modelo de KNN usarás — `KNeighborsClassifier` o `KNeighborsRegressor` — y por qué?

_(Tu respuesta aquí)_

## Paso 3 — Columnas numéricas continuas vs. discretas/categóricas

Define dos listas de nombres de columnas (sin incluir `House_Price`):

- `num_cols`: columnas numéricas continuas → se escalan con `StandardScaler`. (`Square_Footage`, `Year_Built`, `Lot_Size`)
- `cat_cols`: columnas discretas de baja cardinalidad tratadas como categóricas → se codifican con `OneHotEncoder`. (`Num_Bedrooms`, `Num_Bathrooms`, `Garage_Size`, `Neighborhood_Quality`)

In [ ]:
# TODO: completa las dos listas
num_cols = []
cat_cols = []


## Paso 4 — Partición de datos

- Crea `X` (todas las columnas de `df` excepto `House_Price`) e `y` (`House_Price`).
- Usa `train_test_split` con `test_size=0.2` y `random_state=RANDOM_STATE`.
- Nombra las variables `X_train`, `X_test`, `y_train`, `y_test`.

In [ ]:
# TODO: separa X, y y haz el split


## Paso 5 — ColumnTransformer + Pipeline con KNeighborsRegressor

- Construye un `ColumnTransformer` llamado `preprocessor` con dos pasos:
  - `("num", StandardScaler(), num_cols)`
  - `("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)`
- Construye un `Pipeline` llamado `pipe` con dos pasos: `("pre", preprocessor)` y `("knn", KNeighborsRegressor())`.
- Entrena `pipe` sobre `X_train`, `y_train`.

> Nota: usa `sparse_output=False` en el `OneHotEncoder` — KNN con `p=3` (Minkowski) no soporta entradas dispersas (*sparse*).

In [ ]:
# TODO: arma el ColumnTransformer y el Pipeline, y entrénalo sobre X_train, y_train
preprocessor = None
pipe = None


### ✅ Checkpoint 1

In [ ]:
# --- CHECKPOINT 1 ---
assert isinstance(pipe, Pipeline), "pipe debe ser un sklearn Pipeline"
assert list(pipe.named_steps.keys()) == ["pre", "knn"], "El pipeline debe tener los pasos 'pre' y 'knn' en ese orden"

sample_pred = pipe.predict(X_test.head(5))
assert sample_pred.shape == (5,), "predict sobre 5 filas debería devolver 5 valores"

print("✅ Checkpoint 1 OK — el pipeline entrena y predice correctamente")


## Paso 6 — Variar `n_neighbors` y `p` con un `for` anidado

Todavía no hemos visto `GridSearchCV`, así que esta búsqueda se hace "a mano" con un `for` anidado:

- Para cada combinación de `n_neighbors` en `[3, 5, 7, 9, 11, 15]` y `p` en `[1, 2, 3]` (parámetro de distancia Minkowski: `p=1` Manhattan, `p=2` Euclidiana, `p=3`):
  1. Construye un pipeline nuevo (`preprocessor` + `KNeighborsRegressor(n_neighbors=..., p=...)`).
  2. Calcula su desempeño de validación cruzada con `cross_val_score(pipe_i, X_train, y_train, cv=5, scoring="r2")`.
  3. Guarda el promedio de esos 5 scores junto con `n_neighbors` y `p` en una lista.
- Con esa lista arma un DataFrame llamado `cv_results` con columnas `n_neighbors`, `p`, `mean_r2`.
- A partir de `cv_results`, encuentra la fila con mejor `mean_r2` y guarda `best_params` (diccionario `{"n_neighbors": ..., "p": ...}`) y `best_score` (su `mean_r2`).

In [ ]:
# TODO: recorre las combinaciones de n_neighbors y p con un for anidado,
# calcula cross_val_score para cada una y guarda los resultados

results = []
for n_neighbors in [3, 5, 7, 9, 11, 15]:
    for p in [1, 2, 3]:
        pass  # completa aquí: arma el pipeline, corre cross_val_score, guarda el resultado en results

cv_results = pd.DataFrame(results)

# TODO: encuentra la mejor combinación a partir de cv_results
best_params = None
best_score = None


### ✅ Checkpoint 2

In [ ]:
# --- CHECKPOINT 2 ---
assert cv_results.shape[0] == 6 * 3, f"Se esperaban 18 combinaciones, hay {cv_results.shape[0]}"
assert {"n_neighbors", "p", "mean_r2"}.issubset(cv_results.columns), \
    "cv_results debe tener las columnas n_neighbors, p y mean_r2"
assert best_score > 0.75, f"El mejor R² de CV ({best_score:.3f}) es más bajo de lo esperado, revisa tu pipeline"

print(f"✅ Checkpoint 2 OK — mejor R² de CV: {best_score:.3f} con {best_params}")


## Paso 7 — Gráfico: métrica de CV vs. `n_neighbors` por cada `p`

Usa `cv_results` para graficar `mean_r2` en el eje Y, `n_neighbors` en el eje X, y una curva distinta por cada valor de `p`.

In [ ]:
# TODO: arma el gráfico (una curva por cada valor de p)


## Paso 8 — Evaluación en el conjunto de prueba

- Con `best_params`, entrena un pipeline final (`best_model`) sobre `X_train`, `y_train`.
- Predice sobre `X_test` y calcula `MAE`, `RMSE` y `R²` en las variables `mae`, `rmse`, `r2`.

In [ ]:
# TODO: entrena el modelo final con best_params sobre X_train, y_train
best_model = None

# TODO: predice sobre X_test y calcula mae, rmse, r2
mae = None
rmse = None
r2 = None


## Paso 9 — Tabla comparativa por valor de `p`

Para cada valor de `p` en `[1, 2, 3]`:

1. Toma su mejor `n_neighbors` según `cv_results` (el que maximiza `mean_r2` para ese `p`).
2. Entrena un pipeline con esos parámetros sobre `X_train`, `y_train`.
3. Evalúa sobre `X_test` (MAE, RMSE, R²).

Junta todo en un DataFrame llamado `comparison_table` con columnas `p`, `best_n_neighbors`, `MAE`, `RMSE`, `R2`.

In [ ]:
# TODO: construye la tabla comparativa
comparison_table = None


### ✅ Checkpoint 3 (final)

In [ ]:
# --- CHECKPOINT 3 (final) ---
assert r2 > 0.80, f"R² en test ({r2:.3f}) por debajo de 0.80 — revisa preprocesamiento y búsqueda de parámetros"
assert list(comparison_table["p"]) == [1, 2, 3], "comparison_table debe tener una fila por cada p en [1, 2, 3]"
assert {"p", "best_n_neighbors", "MAE", "RMSE", "R2"}.issubset(comparison_table.columns), \
    "Faltan columnas en comparison_table"

print(f"✅ Checkpoint 3 OK — R²={r2:.3f} en test, tabla comparativa completa")
comparison_table


## Paso 10 — Conclusión

En una celda de markdown (mínimo 80 palabras): **¿qué valor de `p` funcionó mejor y por qué creen que pasó eso con este dataset?**

_(Tu respuesta aquí)_